<a href="https://colab.research.google.com/github/prakashgoud421/FragmentTransaction/blob/fragment/gpuchat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Default title text
!pip install -q llama-cpp-python langchain gradio sentence-transformers chromadb PyPDF2 langchain langchain-community pypdf transformers accelerate bitsandbytes sentence-transformers

In [ ]:
!apt-get install -y cmake build-essential
!pip install --upgrade pip
!pip install typing-extensions


In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
%cd /content/llama.cpp

# Clean up any old builds just in case
!rm -rf build
!mkdir build
%cd build

# ✅ Use the correct build flag
!cmake -DGGML_CUDA=on ..

# Build the project
!cmake --build . --config Release --parallel





In [ ]:
%cd /content/llama.cpp/bindings/python
!pip install .


In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())
!nvidia-smi


In [5]:
import os
import hashlib
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from llama_cpp import Llama
from langchain_community.llms import LlamaCpp


In [6]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
MODEL_PATH = "/content/drive/MyDrive/models/mistral-7b-instruct-v0.1.Q4_K_M.gguf"
MODEL_PATH2 = "/content/drive/MyDrive/models/llama-2-7b-chat.Q4_K_M.gguf"
UPLOAD_DIR = "/content/uploaded_pdfs"
DB_DIR = "/content/vector_db"

os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(DB_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
llm = Llama(
    model_path="/content/drive/MyDrive/models/llama-2-7b-chat.Q4_K_M.gguf",
    n_ctx=2048,
    n_gpu_layers=40,  # you can adjust based on available GPU
    verbose=True
)


In [20]:

print("Model exists:", os.path.exists(MODEL_PATH2))

Model exists: False


In [ ]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
qa_chain = None

In [ ]:
response = llm("Q: Who is the Prime Minister of India?\nA:", max_tokens=64)
print(response["choices"][0]["text"])

In [ ]:
model_path = "/content/drive/MyDrive/models/mistral-7b-instruct-v0.1.Q4_K_M.gguf"
llm = LlamaCpp(
    model_path= model_path,  # use a GPU-friendly one
    n_gpu_layers=-1,      # Enable GPU acceleration
    n_ctx=2048,
    temperature=0.7,
    max_tokens=512,
    verbose=True,
)

In [7]:
def get_md5(file_path):
    with open(file_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

In [8]:
def load_or_create_vectorstore(pdf_path):
    file_hash = get_md5(pdf_path)
    persist_path = os.path.join(DB_DIR, file_hash)

    if os.path.exists(persist_path):
        db = Chroma(persist_directory=persist_path, embedding_function=embedding)
    else:
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        docs = splitter.split_documents(documents)
        db = Chroma.from_documents(docs, embedding, persist_directory=persist_path)
        db.persist()
    return db



In [9]:

def upload_file(file_path):
    global qa_chain

    if not os.path.exists(file_path):
        return "❌ File not found."

    try:
        db = load_or_create_vectorstore(file_path)
        retriever = db.as_retriever()

        # Limit token size for llama-cpp compatibility
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=retriever,
            return_source_documents=True,
            chain_type_kwargs={"verbose": True}  # Optional: for debugging
        )

        return f"✅ Uploaded and ready: {os.path.basename(file_path)}"

    except Exception as e:
        return f"❌ Error processing file: {str(e)}"




In [10]:
def ask_question(question):
    if not qa_chain:
        return "❗ Please upload a PDF first."
    output = qa_chain.invoke({"query": question})
    return output["result"]



In [ ]:
!nvidia-smi


In [ ]:

with gr.Blocks() as demo:
    gr.Markdown("## 📄 Chat with your PDF (Mistral + LangChain)")
    file_input = gr.File(label="Upload PDF", type="filepath", file_types=[".pdf"])
    upload_button = gr.Button("Upload PDF")
    upload_status = gr.Textbox(label="Upload Status")

    question_input = gr.Textbox(label="Ask a question")
    answer_output = gr.Textbox(label="Answer")

    upload_button.click(upload_file, inputs=file_input, outputs=upload_status)
    question_input.submit(ask_question, inputs=question_input, outputs=answer_output)

demo.launch(debug=True, share=True, inline=True)